<center>
<h1><strong>Aprendizaje automático e Inteligencia Artificial</strong></h1>
<h2><strong>Especialización en Ciencia de Datos e Inteligencia Artificial - ECDIA</strong></h2>
<h3>Universidad Tecnológica - UTEC</h3>
 <h3><strong>Nombres:</strong> Florencia Barrios, Florencia Lucas, María Martinote</h3>
</center>

#**1- Contextualización de la Problemática**

### 🗂️ Descripción del Dataset


![](https://images.pexels.com/photos/681331/pexels-photo-681331.jpeg?cs=srgb)

El presente dataset contiene información de precio de apartamentos segun varias caracteristicas. La información se obtuvo mediante web scraping de las principales paginas inmobiliarias de Uruguay (Mercado Libre y Gallito).
El periodo de extracción de datos se encuentra entre 2023-2025.

## **Variables del dataset original y tipo de datos**

 *  NEIGHBORHOOD: Localidad (Object)
 *  COVERED_AREA: Área cubierta (Float)
 *  LISTING_TYPE_ID: Tipo de publicación en Meli (Object)
 *  ADDRESS_CITY_NAME: Localidad (Object)
 *  ROOMS: Cantidad de habitaciones (Int)
 *  WITH_VIRTUAL_TOUR: Indicador de tour virtual (Object)
 *  HAS_AIR_CONDITIONING: Indicador de aire acondicionado (Object)
 *  PROCESS_DATE: Fecha de proceso (Object)
 *  TOTAL_AREA: Área total (Int)
 *  DESCRIPTION: Descripción (Object)
 *  ITEM_CONDITION: Estado del inmueble (Object)
 *  PROCESS_DATE: Fecha de proceso (Object)
 *  ADDRESS_STATE: Dirección (Object)
 *  ADDRESS: Departamento (Object)
 *  CONDITION: Estado del inmueble (Object)
 *  TITLE: Título de la publicación (Object)
 *  PRICE: Precio de venta del apartamento (Object)
 *  ADDRESS_LINE: Dirección (Object)
 *  CATEGORY_ID: ID Meli (Object)
 *  ORIGEN: Página de origen (Object)
 *  SITE_ID: ID del sitio web (Meli) (Object)
 *  HAS_TELEPHONE_LINE: Indicador de teléono (Object)
 *  ID: ID Meli (Object)
 *  DESCRIPTION: Descripción (Object)
 *  FULL_BATHROOMS: Cantidad de baños (Float)
 *  OPERATION: Tipo de operación (Object)
 *  BEDROOMS: Cantidad de dormitorios (Float)
 *  CURRENCY_ID: Moneda (Object)
 *  PROPERTY_TYPE: Tipo de inmueble (Object)
 *  POSITION: Posición del inmueble (Object)
 *  ARTICLE_ID: ID Gallito (Float)
 *  GARAGE: Cantidad de garages (Int)
 *  CONSTRUCTION_YEAR: Año de construcción (Float)


### **Objetivo del Análisis**

Predecir el precio de apartamentos en Uruguay a partir de características y atributos obtenidos mediante webscrapping de las principales paginas inmobiliarias.

## **Preguntas Clave**

Algunas preguntas guía para el análisis:

- ¿Cuáles son las principales variables que inciden en el precio?
- ¿Hay diferencias en el precio promedio entre Meli y Gallito?
- ¿Cuáles son las localidades más caras y más baratas?
- ¿Existe multicolinealidad entre las variables independientes?



### **Importación de librerías y paquetes**

In [ ]:
%pip install optuna

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import optuna

# To build models for prediction
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor,BaggingRegressor
from sklearn.tree import plot_tree
from sklearn.tree import export_text
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_percentage_error as mape


# To encode categorical variables
from sklearn.preprocessing import LabelEncoder

# For tuning the model
from sklearn.model_selection import GridSearchCV

# To check model performance
from sklearn.metrics import make_scorer,mean_squared_error, r2_score, mean_absolute_error

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# configuración para mostrar todas las columnas
pd.set_option('display.max_columns', None)

In [ ]:
# configuración para quitar notación científica
pd.set_option('display.float_format', lambda x: '%.2f' % x)

#**2- Limpieza y procesamientos de datos**

### **Importación de datos**

Los pasos a continuación quedan comentados para tomar directamente el archivo desduplicado (más liviano)

In [ ]:
#datos = pd.read_excel('/content/drive/MyDrive/Proyecto Aptos/ventas_aptos.xlsx')

In [ ]:
# Generamos una copia para no modificar la base original
#df = datos.copy()

## **Overview**

In [ ]:
# Revisión de primeras filas
#df.head()

In [ ]:
# Revisión de últimas filas
#df.tail()

**Revisamos las dimensiones de la base**

In [ ]:
#df.shape

**Observaciones:**

El dataset tiene 550,428 filas y 33 columnas.

### **Tratamiento de Duplicados**

Antes de comenzar con el análisis exploratorio, debemos asegurarnos de que no haya registros duplicados. Nuestra unidad de análisis son las publicaciones de apartamentos en venta en Uruguay, extraídas de los portales Meli (Mercado Libre) y Gallito.

Identificamos dos posibles tipos de duplicados en el dataset:

- **Duplicados exactos**: filas que son completamente idénticas en todas sus columnas, representando un procesamiento duplicado del mismo registro en el mismo momento.

- **Duplicados temporales**: publicaciones que fueron procesadas (extraídas) en distintos momentos. Estas filas son idénticas en todas las características del apartamento (ej. área, precio, barrio), pero difieren en las variables de FECHA de extracción.

Para detectar y eliminar ambos tipos, primero eliminamos los duplicados exactos. Posteriormente, para detectar los duplicados temporales, realizamos una búsqueda de duplicados excluyendo las variables de fecha (PROCESS_DATE, PROCESS_DATE.1) de la comparación.

In [ ]:
# Count de duplicados exactos
#duplicados_exactos = df[df.duplicated()]
#print(f"Registros duplicados exactos: {len(duplicados_exactos)}")

In [ ]:
# Contar duplicados temporales (excluyendo las columnas de fecha)
#duplicados_temporales = df.drop(columns=['PROCESS_DATE', 'PROCESS_DATE.1']).duplicated().sum()

#print(f"Registros duplicados temporales: {duplicados_temporales}")

In [ ]:
370792/550428

**Observaciones**:

- El 67% de los registros son duplicados (mismas publicaciones que se procesaron en distintas fechas).
- Eliminamos ambos tipos de  duplicados.

In [ ]:
# Eliminamos duplicados exactos
#df = df.drop_duplicates()

In [ ]:
# Eliminamos duplicados temporales
#df = df.drop_duplicates(subset=df.columns.difference(['PROCESS_DATE', 'PROCESS_DATE.1']))

In [ ]:
#df.shape

In [ ]:
#df.ORIGEN.value_counts()

In [ ]:
# Guardamos base desduplicada (mas liviada)
#df.to_excel('/content/drive/MyDrive/Proyecto Aptos/ventas_aptos_desduplicada.xlsx', index=False)

Empezar a correr desde acá para traer la base ya desduplicada

In [ ]:
datos = pd.read_excel('/content/drive/MyDrive/ventas_aptos_desduplicada.xlsx')

In [ ]:
# Generamos una copia para no modificar la base original
df = datos.copy()

**Conociendo el conjunto de datos**

In [ ]:
df.info()

**Observaciones:**
- Las variables numéricas son: COVERED_AREA, ROOMS, TOTAL_AREA, FULL_BATHROOMS, BEDROOMS, ARTICLE_ID, GARAGE, CONSTRUCTION_YEAR. El resto de las variables son de tipo object.
- Hay pocas variables sin missings: ROOMS, TITLE, PRICE, ORIGEN, OPERATION, CURRENCY_ID, PROPERTY_TYPE, GARAGE.
- PRICE es nuestra variable objetivo, es importante contar con todos los datos en este caso.
- Hay variables que son object y deberían ser numéricas, por ejemplo: PRICE.

### **Resumen de variables numéricas**

In [ ]:
# Pasamos PRICE a numérica
df['PRICE'] = df['PRICE'].astype(str).str.replace('[.,]', '', regex=True)
df['PRICE'] = pd.to_numeric(df['PRICE'])

In [ ]:
df.describe().T

**Observaciones:**
- Hay varios datos incoherentes: ROOMS negativos, máximos muy elevados en ROOMS, BEDROOMS, GARAGE y años erroneos en CONSTRUCTION_YEAR.
- Hay valores atípicos que impactan en los promedios en variables como COVERED_AREA y TOTAL_AREA. En la etapa de análisis univariado podremos verificar outliers.
- ARTICLE_ID no tiene sentido tratarlo como numérico porque es un dato descriptivo.
- Los cuartiles de TOTAL_AREA y COVERED_AREA son muy similares, son variables que parecerían estar correlacionadas.
- La mediana del predio de los apartamentos es 169 mil USD. Se deben hacer un tratamiento de outliers en esta variable también.

### **Resumen de variables no numéricas**

In [ ]:
# Pasamos a categorica ARTICLE_ID
df['ARTICLE_ID'] = df['ARTICLE_ID'].astype('category')

In [ ]:
df.describe(exclude = 'number').T

In [ ]:
# Aptos de punta del este
(40139+ 14368)/179636

In [ ]:
df.ORIGEN.value_counts()

In [ ]:
df.ORIGEN.value_counts(normalize=True)

**Observaciones:**

- El 30% de los apartamentos son de Punta del Este.
- El 70% de los casos son publicaciones del Gallito, el resto son de Mercado Libre.
- Se podría deducir que las variables con 52231 datos son de los apartamentos de Mercado Libre (Ej. ADDRESS_CITY_NAME) y aquellas con 127405 datos son del Gallito (Ej. NEIGHBORHOOD).
- La gran mayoría de las publicaciones son en dólares.
- Hay variables que no aportan valor al análisis como OPERATION (son todas "Venta"), PROPERTY_TYPE (son todas "Apartamento"), SITE_ID, CATEGORY_ID y ID (parecen ser códigos de Mercado Libre)
- Vemos que hay 52231 registros con el ID de Meli y todos son únicos.
- Para el Gallito (ARTICLE_ID) hay 127405 registros y solo 93274 son únicos. Reviamos los duplicados por ID.


**Revisamos duplicados por ARTICLE_ID**

In [ ]:
# A ver los registros duplicados de article_id ordenados por article_id
df_duplicados = df[df.duplicated(subset='ARTICLE_ID', keep=False)].sort_values(by='ARTICLE_ID')

In [ ]:
df_duplicados.shape

In [ ]:
# guardo en csv df_duplicados
#df_duplicados.to_csv('/content/drive/MyDrive/Proyecto Aptos/df_duplicados.csv', index=False)

**Observaciones:**

Notamos que la publicación es la misma y en general hay pequeñas diferencias (ej. Descripción, Título, etc.). Decidimos eliminar estos duplicados.

In [ ]:
# Eliminamos duplicados por ARTICLE_ID y nos quedamos con el registro mas reciente (PROCESS_DATE)
# Primero separamos los registros con ARTICLE_ID nulo
df_null_article_id = df[df['ARTICLE_ID'].isnull()]

# Luego procesamos los registros con ARTICLE_ID no nulo
df_not_null_article_id = df[df['ARTICLE_ID'].notnull()].sort_values(by='PROCESS_DATE', ascending=True).drop_duplicates(subset='ARTICLE_ID', keep='last')

# Concatenamos los dos dataframes
df = pd.concat([df_null_article_id, df_not_null_article_id])

In [ ]:
df.shape

**Revisamos los valores de las variables categóricas**

In [ ]:
# Generamos lista con las variables categóricas de interés
cat_col = ['NEIGHBORHOOD', 'LISTING_TYPE_ID', 'ADDRESS_CITY_NAME', 'ITEM_CONDITION', 'CONDITION', 'CURRENCY_ID', 'POSITION', 'ADDRESS_STATE']

# Para cada variable traemos categorías y su frecuencia
for column in cat_col:
    print(df[column].value_counts())

    print('-' * 50)

**Observaciones:**

- A simple vista NEIGHBORHOOD/ADDRESS_CITY_NAME, CONDITION/ITEM_CONDITION, POSITION nos parecen variables relevantes para el análisis.
- Habría que tratar el precio en función de CURRENCY_ID.

## **Tratamiento de datos**

### **Tratamiento de precio según moneda**

In [ ]:
df['PRICE'].describe()

**Observaciones:**

- Vemos un dato atípico, erroneo como máximo (444444444.00).
- La media se encuentra en 264.054 USD.
- La mediana es de 170.000 USD.
- También hay datos erroneos en el extremo inferior (mínimo de 1).

In [ ]:
# Tabla cruzada de currency y origen
pd.crosstab(df['CURRENCY_ID'], df['ORIGEN'], dropna=False)

In [ ]:
# Revisamos casos con currency distinta a USD
reviso_pesos= df[df['CURRENCY_ID'] != 'USD']

In [ ]:
# Describe de price en reviso_pesos
reviso_pesos['PRICE'].describe()

In [ ]:
# ordenar reviso_pesos descendente por precio
reviso_pesos.sort_values(by='PRICE', ascending=False).head()

**Observaciones:**

- El valor mínimo es incoherente
- Analizando el caso de 3900000, parecería ser un error de tipeo
- Entendemos que los casos en pesos tienen los precios en dólares ya que no tiene sentido que el precio de venta de un apartamento sea $155.892 (la media de los casos en pesos)

In [ ]:
# eliminamos currency dado que los precios estan todos en dolares
df = df.drop(['CURRENCY_ID'], axis=1)

### **Transformación de tipos de datos**

**Revisión de dummies para verificar valores**

In [ ]:
df.WITH_VIRTUAL_TOUR.value_counts()

In [ ]:
df.HAS_AIR_CONDITIONING.value_counts()

In [ ]:
df.HAS_TELEPHONE_LINE.value_counts()

In [ ]:
# Pasamos a dummies las variables analizadas
df['WITH_VIRTUAL_TOUR'] = df['WITH_VIRTUAL_TOUR'].map({'Sí': 1, 'No': 0})
df['HAS_AIR_CONDITIONING'] = df['HAS_AIR_CONDITIONING'].map({'Sí': 1, 'No': 0})
df['HAS_TELEPHONE_LINE'] = df['HAS_TELEPHONE_LINE'].map({'Sí': 1, 'No': 0})

### **Tratamiento de fechas**

A partir del describe de variables categóricas notamos que PROCESS_DATE corresponde a las publicaciones del gallito y tiene formato yyyy-mm-dd y PROCESS_DATE.1 corresponde a las de meli y tiene formato dd/mm/yyyy. Verificamos a continuación.

In [ ]:
# Revisión del formato de fecha según el origen
meli=df[df['ORIGEN']=='meli']
gallito=df[df['ORIGEN']=='gallito']

In [ ]:
print(len(meli))
print(len(gallito))

In [ ]:
# Revisamos missings de PROCESS_DATE en meli
print(meli['PROCESS_DATE'].isnull().sum())
print(meli['PROCESS_DATE.1'].isnull().sum())

In [ ]:
# Revisamos missings de PROCESS_DATE en gallito
print(gallito['PROCESS_DATE'].isnull().sum())
print(gallito['PROCESS_DATE.1'].isnull().sum())

In [ ]:
# Dejamos process date y process date 1 en el mismo formato de date
df['PROCESS_DATE'] = pd.to_datetime(df['PROCESS_DATE'], format='%Y-%m-%d')
df['PROCESS_DATE.1'] = pd.to_datetime(df['PROCESS_DATE.1'], format='%d/%m/%Y')

In [ ]:
# Generamos una nueva variable de fecha. Si el origen es meli entonces usamos PROCESS_DATE.1, si el origen en gallito usamos PROCESS_DATE
df['FECHA'] = np.where(df['ORIGEN'] == 'meli', df['PROCESS_DATE.1'], df['PROCESS_DATE'])

In [ ]:
# Revisamos que no tenga missings
df['FECHA'].isnull().sum()

In [ ]:
df['FECHA'].describe()

In [ ]:
df['FECHA'].value_counts()

In [ ]:
# decribe de fecha segun origen
df.groupby('ORIGEN')['FECHA'].describe()

**Observaciones:**

Las publicaciones de Gallito son solo de 2023 mientras que las de Meli van desde marzo de 2023 hasta marzo de 2025.

### **Unificación de variables duplicadas**

In [ ]:
# Revisamos missings de CIUDAD en meli
print(meli['ADDRESS_CITY_NAME'].isnull().sum())
print(meli['NEIGHBORHOOD'].isnull().sum())

In [ ]:
# Revisamos missings de CIUDAD en gallito
print(gallito['ADDRESS_CITY_NAME'].isnull().sum())
print(gallito['NEIGHBORHOOD'].isnull().sum())

In [ ]:
# Revisamos missings de DESCRIPCION en meli
print(meli['DESCRIPTION'].isnull().sum())
print(meli['DESCRIPTION.1'].isnull().sum())

In [ ]:
# Revisamos missings de DESCRIPCION en gallito
print(gallito['DESCRIPTION'].isnull().sum())
print(gallito['DESCRIPTION.1'].isnull().sum())

In [ ]:
# Revisamos missings de AREA en meli
print(meli['COVERED_AREA'].isnull().sum())
print(meli['TOTAL_AREA'].isnull().sum())

In [ ]:
# Revisamos missings de AREA en gallito
print(gallito['COVERED_AREA'].isnull().sum())
print(gallito['TOTAL_AREA'].isnull().sum())

In [ ]:
# tabla cruzada de missings de construction year y de condition item
pd.crosstab(df['CONSTRUCTION_YEAR'].isnull(), df['ITEM_CONDITION'].isnull(), dropna=False, normalize=True)

In [ ]:
# revisar missings de construction year
df[df['CONSTRUCTION_YEAR'].isnull()].head()

In [ ]:
# Revisamos missings de CONDICION en meli
print(meli['CONDITION'].isnull().sum())
print(meli['ITEM_CONDITION'].isnull().sum())

In [ ]:
# Revisamos missings de CONDICION en gallito
print(gallito['CONDITION'].isnull().sum())
print(gallito['ITEM_CONDITION'].isnull().sum())

In [ ]:
# tabla cruzada de condicion e item condition incluyendo missings
pd.crosstab(df['CONDITION'], df['ITEM_CONDITION'], dropna=False)

**Observaciones:**

- CONDITION solo tiene los valores "new" y "used" que ya vienen en ITEM_CONDITION como "nuevo" y "usado". Entonces podríamos prescindir de la variable CONDITION.
- A partir de las tablas anteriores confirmamos que NEIGHBORHOOD tiene datos solo para meli y ADDRESS_CITY_NAME solo del gallito y que DESCRIPTION es la variable de meli y DESCRIPTION.1 es la del gallito. Eliminamos variables duplicadas.

In [ ]:
# NEIGHBORHOOD y ADDRESS_CITY_NAME
df['LOCALIDAD'] = np.where(df['ORIGEN'] == 'meli', df['ADDRESS_CITY_NAME'], df['NEIGHBORHOOD'])

# DESCRIPTION y DESCRIPTION.1
df['DESCRIPCION'] = np.where(df['ORIGEN'] == 'meli', df['DESCRIPTION'], df['DESCRIPTION.1'])

### **Eliminamos variables que no aportan valor**

In [ ]:
# Eliminamos variables duplicadas por origen
df = df.drop(['PROCESS_DATE', 'PROCESS_DATE.1', 'ADDRESS_CITY_NAME', 'NEIGHBORHOOD', 'DESCRIPTION', 'DESCRIPTION.1', 'CONDITION'], axis=1)

# Eliminamos variables descriptivas que no aportan valor al análisis (sin variabilidad o identificadores de origen)
df = df.drop(['OPERATION', 'PROPERTY_TYPE', 'SITE_ID', 'CATEGORY_ID', 'ID', 'ARTICLE_ID'], axis=1)

# Eliminamos direcciones (mantenemos unicamente localidad para el análisis)
df = df.drop(['ADDRESS', 'ADDRESS_LINE'], axis=1)

### **Tratamiento de Missings**

In [ ]:
# Revisamos el porcentaje de missings por variable
df.isnull().sum()*100/len(df)

**Observaciones:**

Hay variables que superan el 60% de missings. Las analizamos más en detalle:

- Algunas no parecen relevantes para predecir el precio entonces decidimos eliminarlas: LISTING_TYPE_ID, WITH_VIRTUAL_TOUR, HAS_AIR_CONDITIONING, HAS_TELEPHONE_LINE

- TOTAL_AREA la eliminamos porque ya está COVERED_AREA que es similar y tiene datos para casi el 100% del dataset.

- ADRESS_STATE la eliminamos porque este mismo dato se puede tomar de LOCALIDAD que no tiene missings.

- Identificamos otras variables con un alto porcentaje de missings (CONSTRUCTION_YEAR, POSITION y ITEM_CONDITION) que vamos a analizar en una etapa posterior.

In [ ]:
# Eliminamos las variables
df = df.drop(['LISTING_TYPE_ID', 'WITH_VIRTUAL_TOUR', 'HAS_AIR_CONDITIONING', 'HAS_TELEPHONE_LINE', 'TOTAL_AREA', 'ADDRESS_STATE'], axis=1)

### **Duplicados**

Revisamos de vuelta duplicados después de eliminar variables.

In [ ]:
# Análisis de valores duplicados

duplicados = df.duplicated().sum()
print(f"Registros duplicados: {duplicados}")

Encontramos 2859 filas completamente duplicadas. Las eliminamos.

In [ ]:
# eliminamos duplicados
df = df.drop_duplicates()

In [ ]:
df.shape

# **3- Análisis Exploratorio**

## **Análisis Univariado**

**Variables continuas**

In [ ]:
df.columns

In [ ]:
# Generamos lista con variables continuas
num_col = ['PRICE', 'COVERED_AREA', 'CONSTRUCTION_YEAR']

In [ ]:
for col in num_col:
    print(col)
    if df[col].dtype in ['int64', 'float64']:
        print('Skew :',round(df[col].skew(),2))
        plt.figure(figsize=(15,4))
        plt.subplot(1,2,1)
        df[col].hist(bins=10, grid=False)
        plt.ylabel('count')
        plt.subplot(1,2,2)
        sns.boxplot(x=df[col])
        plt.show()

**Observaciones:**

- Todas tiene valores atípicos. Revisamos la base y eliminamos estos datos para hacer de nuevo el análisis gráfico.

**Precio**

In [ ]:
# Vemos percentiles de precio
df['PRICE'].describe(percentiles=[0.005, 0.01, 0.05, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.975, 0.99, 0.995])

In [ ]:
# Generamos una nueva copia para no eliminar outliers de la base original
df_clean = df.copy()

In [ ]:
# Filtramos el DataFrame para quedarnos con los valores de PRICE entre el percentil 0.5 y 500.000 USD
df_clean = df_clean[(df_clean['PRICE'] >= df_clean['PRICE'].quantile(0.005)) & (df_clean['PRICE'] <= 500000)]

In [ ]:
df_clean.shape

**Covered Area**

In [ ]:
# Vemos percentiles de COVERED_AREA
df_clean['COVERED_AREA'].describe(percentiles=[0.01, 0.05,0.08,  0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.975, 0.995])

In [ ]:
# Filtramos el DataFrame para quedarnos con los valores de COVERED_AREA entre el percentil 8 y 97.5
df_clean = df_clean[(df_clean['COVERED_AREA'] >= df_clean['COVERED_AREA'].quantile(0.08)) & (df_clean['COVERED_AREA'] <= df_clean['COVERED_AREA'].quantile(0.975))]

In [ ]:
df_clean.shape

**Construction Year**

In [ ]:
# Vemos percentiles de precio
df_clean['CONSTRUCTION_YEAR'].describe(percentiles=[0.01, 0.015, 0.05,0.075,  0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.975, 0.99, 0.995])

In [ ]:
# Filtramos años de construcción entre 1900 y 2026, manteniendo los missings
df_clean = df_clean[((df_clean['CONSTRUCTION_YEAR'] >= 1900) & (df_clean['CONSTRUCTION_YEAR'] <= 2026)) | df_clean['CONSTRUCTION_YEAR'].isna()]

**Graficamos variables continuas sin atípicos**

In [ ]:
# volvemos a generar los gráficos con las variables limpias
for col in num_col:
    print(col)
    if df_clean[col].dtype in ['int64', 'float64']:
        print('Skew :',round(df_clean[col].skew(),2))
        plt.figure(figsize=(15,4))
        plt.subplot(1,2,1)
        df_clean[col].hist(bins=10, grid=False)
        plt.ylabel('count')
        plt.subplot(1,2,2)
        sns.boxplot(x=df_clean[col])
        plt.show()

**Observaciones:**

- El precio tiene una distribución sesgada a la derecha (asimetría postivia). Se espera que los precios se concentren en valores más bajos.
- El área cubierta presenta una distribución similar al precio.
- El año de construcción tiene una asimetría negativa (la mayoría de los apartamentos son de menor antiguedad).

In [ ]:
# volvemos a renombrar la base
df = df_clean

**Variables Discretas**

**ROOMS**

In [ ]:
df.ROOMS.value_counts()

In [ ]:
# ordenamos descendente por rooms y head
df.sort_values(by='ROOMS', ascending=False).head()

In [ ]:
# Eliminamos datos incoherentes o atípicos: negativos y mayores a 5
df = df[df['ROOMS'] >= 0]
df = df[df['ROOMS'] <= 5]

In [ ]:
# Gráfico de barras
sns.countplot(x = 'ROOMS', data = df, color = 'blue',
              order = sorted(df['ROOMS'].unique()));

In [ ]:
# tabla cruzada de rooms con origen
pd.crosstab(df['ROOMS'], df['ORIGEN'], dropna=False)

Observaciones:
- Rooms no viene en el gallito. Si bien no tenía missings, todos los datos del gallito están en 0.

In [ ]:
# tabla cruzada de rooms y bedrooms filtrando por origen = meli
pd.crosstab(df[df['ORIGEN'] == 'meli']['ROOMS'], df[df['ORIGEN'] == 'meli']['BEDROOMS'], dropna=False)

In [ ]:
# tabla cruzada de rooms y bedrooms filtrando por origen = meli en porcentaje por fila
pd.crosstab(df[df['ORIGEN'] == 'meli']['ROOMS'], df[df['ORIGEN'] == 'meli']['BEDROOMS'], dropna=False, normalize='index')

In [ ]:
# ROOMS tiene la misma información que BEDROOMS. Eliminamos ROOMS
df = df.drop(['ROOMS'], axis=1)

**BEDROOMS**

In [ ]:
df.BEDROOMS.value_counts()

In [ ]:
# Eliminamos datos incoherentes o atípicos: mayores a 5
df = df[df['BEDROOMS'] <= 5]

In [ ]:
# Gráfico de barras
sns.countplot(x = 'BEDROOMS', data = df, color = 'blue',
              order = sorted(df['BEDROOMS'].unique()));

Observaciones:
- La mayoría de los apartamentos tiene 2 dormitorios, le sigue 1, 3, 4 y luego 0.
- Casi no hay apartamentos con 5 o más dormitorios.

**FULL_BATHROOMS**

In [ ]:
df.FULL_BATHROOMS.value_counts()

In [ ]:
# Eliminamos datos incoherentes o atípicos: mayores a 5
df = df[df['FULL_BATHROOMS'] <= 5]

In [ ]:
# Gráfico de barras
sns.countplot(x = 'FULL_BATHROOMS', data = df, color = 'blue',
              order = sorted(df['FULL_BATHROOMS'].unique()));

Observaciones:
- Los datos se concentran en 1, 2 y 3 baños

**GARAGE**

In [ ]:
df.GARAGE.value_counts()

In [ ]:
# Eliminamos datos incoherentes o atípicos: mayores a 3
df = df[df['GARAGE'] <= 3]

In [ ]:
# Gráfico de barras
sns.countplot(x = 'GARAGE', data = df, color = 'blue',
              order = sorted(df['GARAGE'].unique()));

Observaciones:
- La mayoría de los apartamentos tienen 1 o ningun garage.

**Variables Categóricas**

**LOCALIDAD**

In [ ]:
df.LOCALIDAD.value_counts()

Creemos que LOCALIDAD es una variable que aporta valor al modelo, pero requiere mucha limpieza. Nos apoyamos en Gemini para generar una función de limpieza y crear LOCALIDAD_AGRUPADA.

In [ ]:
# quitamos espacios
df['LOCALIDAD'] = df['LOCALIDAD'].str.strip()

In [ ]:
import unicodedata

# --- 2. FUNCIÓN DE LIMPIEZA ---
def limpiar_y_normalizar(texto):
    """
    Función para limpiar y estandarizar los nombres de las localidades.
    """
    # Si es NaN (missing), lo convertimos en 'otros' de inmediato
    if pd.isna(texto):
        return 'otros'

    # Convertir a minúsculas y quitar espacios extra
    texto = str(texto).lower().strip()

    # --- Correcciones manuales específicas ---
    # Aquí puedes añadir todas las que encuentres
    correcciones = {
        'p. ballena': 'punta ballena',
        'b. de carrasco': 'barra de carrasco',
        'punta del este': 'punta del este', # Unifica mayúsculas
        'punta del este': 'punta del este',
    }
    for original, corregido in correcciones.items():
        if texto == original:
            texto = corregido

    # --- Quitar acentos (diacríticos) ---
    # Esto convierte "cordón" en "cordon"
    texto_normalizado = unicodedata.normalize('NFD', texto)
    texto_sin_acentos = "".join(
        [c for c in texto_normalizado if not unicodedata.combining(c)]
    )

    return texto_sin_acentos

# --- 3. APLICAR LA LIMPIEZA ---
# Creamos una nueva columna 'localidad_limpia'
df['localidad_limpia'] = df['LOCALIDAD'].apply(limpiar_y_normalizar)

print("--- Datos Limpios (Conteo) ---")
print(df['localidad_limpia'].value_counts())
print("-" * 30)


# --- 4. AGRUPAR CATEGORÍAS "RARAS" (LONG TAIL) ---

# Define tu umbral.
# Cualquier localidad con MENOS de 'umbral' apariciones será 'otros'.
# En este ejemplo pequeño usamos 3, en tu caso real podría ser 50 o 100.
UMBRAL = 10

# Calcular las frecuencias de las localidades limpias
frecuencias = df['localidad_limpia'].value_counts()

# Identificar las localidades que están por DEBAJO del umbral
localidades_raras = frecuencias[frecuencias < UMBRAL].index

**Municipios**

Creamos una nueva variable "Municipios":
- **Municipio A**: 3 de abril, Barrio Artigas, Cabaña Anaya, Cno. el Tapir, Chimeneas, Condominio 11, El Húmedo, Gori, Jardines de las Torres, Jardines de Paso de la Arena, La Carreta, La Colorada, Las Flores, Las Higueritas, Las Torres, Los Boulevares, Mailhos, Maracaná, Montecarlo, Municipal 18, Nuevo Las Flores, Nuevo Las Torres, Parada Nueva, Parque Lecocq, Parque Tomkinson, Paso de la Arena, Paurú, Punta Espinillo, Santiago Vázquez, Sarandí, Villa Sarandí y Rincón del Cerro. Casco del Cerro, Casabó, Pajas Blancas, Santa Catalina, Cerro Norte, La Boyada, Cerro Oeste y zona rural, Prado Norte, Sayago Oeste, Paso Molino, Belvedere, La Teja, Pueblo Victoria, Tres Ombúes, Nuevo París y Villa Teresa.
- **Municipio B**: Cordón, Palermo, Parque Rodó, Aguada Este, parte de La Comercial y de Tres Cruces, Ciudad Vieja, Centro, Barrio Sur y parte de la Aguada.
- **Municipio C**: Reducto, Bella Vista, Prado, Arroyo Seco, Aguada y Capurro,  Aires Puros, Atahualpa, Prado, Solís, Nueva Savona, Cristóbal Colón y Complejo Habitacional Parque Posada, Goes, Villa Muñoz, Jacinto Vera, Figurita, Reducto, Krüger, Simón Bolívar, Brazo Oriental, parte de la Comercial, Larrañaga y Aguada.
- **Municipio CH**: Punta Carretas, Pocitos, Buceo, Parque Batlle y Villa Dolores, Tres Cruces y parte de Larrañaga, La Blanqueada, Parque Batlle y Buceo
- **Municipio D**: Villa Española, Unión, Pérez Castellanos, Cerrito, Porvenir, Plácido Ellauri, Marconi, Casavalle, Borro, Bonomi, Municipal, Instrucciones, Jardines de Instrucciones, Fraternidad, Cóppola y Las Acacias, Manga, Piedras Blancas, Bola de Nieve, Boizo Lanza, Barrio Franco, Toledo Chico, Transatlántico, Barrio Cirilo, Plus Ultra, Buenos Aires y La Selva.
- **Municipio E**: Carrasco Sur, Carrasco Norte y La Cruz de Carrasco, Buceo, Malvín Nuevo, Malvín y Punta Gorda, Malvín Norte, Unión y La Blanqueada (Este)
- **Municipio F**: Villa García, Manga, Bañados de Carrasco, Las Canteras, Maroñas, Parque Guaraní, Villa Española, Flor de Maroñas, Ituzaingó, Jardines del Hipódromo, Piedras Blancas, Km 16 Cno. Maldonado, Ideal, Industrial, Málaga, Punta de Rieles, Bella Italia, entre otros.
- **Municipio G**: Colón, Villa Colón, Melilla, Abayubá, Cuchilla Pereyra y San Bartolo,  Sayago, Conciliación, Peñarol, Millán y Lecocq, Lavalleja, Prado Chico y Prado Norte.


In [ ]:
# Define the list of localities to search for
localidades_a_buscar = [
    '3 de abril', 'barrio artigas', 'cabaña anaya', 'cno. el tapir', 'chimeneas',
    'condominio 11', 'el humedo', 'gori', 'jardines de las torres',
    'jardines de paso de la arena', 'la carreta', 'la colorada', 'las flores',
    'las higueritas', 'las torres', 'los boulevares', 'mailhos', 'maracana',
    'montecarlo', 'municipal 18', 'nuevo las flores', 'nuevo las torres',
    'parada nueva', 'parque lecocq', 'parque tomkinson', 'paso de la arena',
    'pauru', 'punta espinillo', 'santiago vazquez', 'sarandi', 'villa sarandi',
    'rincon del cerro', 'casco del cerro', 'casabo', 'pajas blancas',
    'santa catalina', 'cerro norte', 'la boyada', 'cerro oeste', 'prado norte',
    'sayago oeste', 'paso molino', 'belvedere', 'la teja', 'pueblo victoria',
    'tres ombues', 'nuevo paris', 'villa teresa', 'cordon', 'palermo',
    'parque rodo', 'aguada este', 'la comercial', 'tres cruces', 'ciudad vieja',
    'centro', 'barrio sur', 'aguada', 'reduto', 'bella vista', 'prado',
    'arroyo seco', 'capurro', 'aires puros', 'atahualpa', 'goes', 'villa munoz',
    'jacinto vera', 'la figurita', 'kruger', 'simon bolivar', 'brazo oriental',
    'larranaga', 'punta carretas', 'pocitos', 'buceo', 'parque batlle',
    'villa dolores', 'la blanqueada', 'villa espanola', 'union',
    'perez castellanos', 'cerrito', 'porvenir', 'placido ellauri', 'marconi',
    'casavalle', 'borro', 'bonomi', 'manga', 'piedras blancas', 'ituzaingo',
    'jardines del hipodromo', 'las canteras', 'maronas', 'parque guarani',
    'flor de maronas', 'bella italia', 'colon', 'villa colon', 'melilla',
    'abayuba', 'cuchilla pereyra', 'san bartolo', 'sayago', 'conciliacion',
    'penarol', 'millan y lecocq', 'lavalleja', 'prado chico', 'carrasco sur',
    'carrasco norte', 'la cruz de carrasco', 'malvin nuevo', 'malvin',
    'punta gorda', 'malvin norte', 'barrio franco', 'toledo chico',
    'transatlantico', 'barrio cirillo', 'plus ultra', 'buenos aires',
    'la selva', 'j. hipodromo', 'jardines de instrucciones', 'fraternidad',
    'coppolla', 'las acacias', 'ideal', 'industrial', 'malaga',
    'punta de rieles', 'villa garcia', 'bañado de carrasco', 'floresta',
    'km 16'
]

# Filter the DataFrame to keep only rows where 'localidad_limpia' is in the list
localidades_encontradas_df = df[df['localidad_limpia'].isin(localidades_a_buscar)]

# Get the unique values from 'localidad_limpia' that are in the list
localidades_encontradas = localidades_encontradas_df['localidad_limpia'].unique()

# Print the found localities
print("Localidades from the list found in 'localidad_limpia':")
print(localidades_encontradas)

In [ ]:
# Mapeo de localidades encontradas con municipios de Montevideo
municipio_mapping = {
    'cordon': 'Municipio B',
    'lezica': 'Municipio G',
    'maronas curva': 'Municipio F',
    'golf': 'Municipio B',
    'nuevo centro': 'Municipio B',
    'parque rodo': 'Municipio B',
    'punta carretas': 'Municipio CH',
    'shopping': 'Municipio CH',
    'villa biarritz': 'Municipio CH',
    'la blanqueada': 'Municipio CH',
    'ciudad vieja': 'Municipio B',
    'centro': 'Municipio B',
    'jacinto vera': 'Municipio C',
    'bella vista': 'Municipio C',
    'pocitos': 'Municipio CH',
    'pocitos nuevo': 'Municipio CH',
    'tres cruces': 'Municipio B',
    'atahualpa': 'Municipio C',
    'nuevo paris': 'Municipio A',
    'malvin': 'Municipio E',
    'parque batlle': 'Municipio CH',
    'sayago': 'Municipio G',
    'aires puros': 'Municipio C',
    'union': 'Municipio D',
    'aguada': 'Municipio B',
    'palermo': 'Municipio B',
    'prado': 'Municipio C',
    'buceo': 'Municipio CH',
    'reduto': 'Municipio C',
    'barrio sur': 'Municipio B',
    'puerto buceo': 'Municipio CH',
    'cerrito': 'Municipio D',
    'cerro': 'Municipio D',
    'villa dolores': 'Municipio CH',
    'goes': 'Municipio C',
    'villa espanola': 'Municipio D',
    'malvin norte': 'Municipio E',
    'la comercial': 'Municipio B',
    'brazo oriental': 'Municipio C',
    'punta gorda': 'Municipio E',
    'paso molino': 'Municipio A',
    'la teja': 'Municipio A',
    'villa munoz': 'Municipio C',
    'penarol': 'Municipio G',
    'colon': 'Municipio G',
    'belvedere': 'Municipio A',
    'piedras blancas': 'Municipio D',
    'perez castellanos': 'Municipio D',
    'capurro': 'Municipio C',
    'ituzaingo': 'Municipio F',
    'arroyo seco': 'Municipio C',
    'larranaga': 'Municipio C',
    'la figurita': 'Municipio C',
    'maronas': 'Municipio F',
    'barrio sur': 'Municipio B',
    'carrasco norte': 'Municipio E',
    'j. hipodromo': 'Municipio F',
    'jardines hipodromo': 'Municipio F',
    'prado norte': 'Municipio G',
    'carrasco sur': 'Municipio E',
    'carrasco': 'Municipio E',
    'san bartolo': 'Municipio G',
    'prado chico': 'Municipio G',
    'malvin nuevo': 'Municipio E',
    'malvín': 'Municipio E',
    'prado chico': 'Municipio G',
    'punta rieles': 'Municipio F',
        'reducto': 'Municipio C',
    'puerto buceo': 'Municipio CH',
    'pocitos nuevo': 'Municipio CH',
    'mercado modelo': 'Municipio D',
    'maronas, curva': 'Municipio F',
    'las acacias' : 'Municipio D',
    'paso de la arena':'Municipio A',
     'cno. carrasco': 'Municipio E',
    'casavalle': 'Municipio D',

}

# Create the 'Municipios' column by mapping the cleaned locality names
df['Municipios'] = df['localidad_limpia'].map(municipio_mapping)

# Fill any remaining NaN values (localities not in the mapping) with 'otros'
df['Municipios'] = df['Municipios'].fillna('otros')

# Display the value counts for the new 'Municipios' column
print(df['Municipios'].value_counts())

In [ ]:
# buscar localidaes con Municipios=otros
localidades_no_encontradas = df[df['Municipios'] == 'otros']['localidad_limpia'].unique()

In [ ]:
# a ver las localidades no encontradas
#localidades_no_encontradas = df[~df['localidad_limpia'].isin(localidades_a_buscar)]['localidad_limpia'].unique()

In [ ]:
print(localidades_no_encontradas)

In [ ]:
# se mapean las localidades no encontradas con el departamento (son localidades del interior del país)
departamentos_mapping = {
    'punta del este': 'Maldonado',
    'laguna de los patos':'Colonia',
    'villa crespo':'Canelones',
    'balneario buenos aires': 'Maldonado',
    'cno. maldonado':'Municipio F',
    'treinta y tres':'Treinta y tres',
    'mata siete':'Canelones',
    'lausana': 'Maldonado',
    'piedras del chileno': 'Maldonado',
    'el vigia': 'Maldonado',
    'punta negra': 'Maldonado',
    'cerro san antonio': 'Maldonado',
    'altos de laguna': 'Maldonado',
    'pan de azucar': 'Maldonado',
    'shangrila':'Canelones',
    'colonia nicolich':'Canelones',
    'jose ignacio': 'Maldonado',
    'garzon': 'Maldonado',
    'laguna garzon': 'Maldonado',
    'playa hermosa': 'Maldonado',
    'piriapolis': 'Maldonado',
    'Municipios': 'Montevideo',
    'montevideo': 'Montevideo',
    'colonia del sacramento': 'Colonia',
    'Colonia del Sacramento': 'Colonia',
    'colonia': 'Colonia',
    'barra de carrasco': 'Canelones',
    'parque miramar': 'Canelones',
    'lagomar': 'Canelones',
    'la paz': 'Canelones',
    'ciudad de la costa': 'Canelones',
    'maldonado': 'Maldonado',
    'manantiales': 'Maldonado',
    'atlantida': 'Canelones',
    'punta ballena': 'Maldonado',
    'pinares': 'Maldonado',
    'paysandu': 'Paysandu',
    'la tahona': 'Canelones',
    'san rafael': 'Maldonado',
    'rincon del indio': 'Maldonado',
    'rincon del indio': 'Maldonado',
    'Canelones': 'Canelones',
    'canelones': 'Canelones',
    'punta del diablo' :'Rocha',
    'La Paloma': 'Rocha',
    'la Paloma': 'Rocha',
    'Pando': 'Canelones',
    'pando': 'Canelones',
    'La Barra':'Maldonado',
    'la barra':'Maldonado',
    'haras del lago':'Canelones',
    'bolivar':'Canelones',
    'san carlos':'Maldonado',
    'playa brava':'Maldonado',
    'playa mansa':'Maldonado',
    'las piedras':'Canelones',
    'rivera':'Rivera',
    'melo': 'Cerro Largo',
    'san jose de carrasco':'Canelones',
    'trinidad':'Flores',
    'mercedes':'Soriano',
    'carmelo':'Colonia',
    'peninsula':'Maldonado',
    'roosevelt':'Maldonado',
    'cantegril':'Maldonado',
    'lomo de la ballena':'Maldonado',
    'bikini':'Maldonado',
    'aidy grill':'Maldonado',
    'laguna del diario':'Maldonado',
    'solanas':'Maldonado',
    'Puerto':'Maldonado',
    'puerto':'Maldonado',
    'la pastora':'Maldonado',
    'paso carrasco':'Canelones',
    'montoya':'Maldonado',
    'otros': 'Otros',
    'paso de carrasco': 'Canelones',
    'solymar': 'Canelones',
    'parque carrasco': 'Canelones',
    'cerro pelado': 'Maldonado',
    'la sonrisa': 'Maldonado',
    'la juanita': 'Maldonado',
    'puerto': 'Maldonado',
    'marconi': 'Municipio D',
    'nueva helvecia': 'Colonia',
    'san jose de mayo': 'San Jose',
    'san Jose': 'San Jose',
    'durazno': 'Durazno',
    'rosario':'Rocha',
    'salto': 'Salto',
    'santiago vazquez' : 'San Jose',
    'manga': 'Municipio D',
    'villa argentina' : 'Canelones',
    'san francisco': 'Maldonado',
    'progreso' : 'Canelones',
    'sauce de portezuelo': 'Maldonado',
    'sauce': 'Maldonado',
    'portezuelo': 'Maldonado',
    'florida' :'Florida',
    'aeropuerto internacional de carrasco' :'Canelones',
    'juan lacaze' :'Colonia',
    'salinas' : 'Canelones',
    'city golf': 'Canelones',
    'santa lucia' : 'San Jose',
    'el pinar': 'Canelones',
    'minas' : 'Minas',
    'artigas': 'Artigas',
    'aguas dulces': 'Rocha',
    'rocha' :'Rocha',
    'el empalme': 'Canelones',
    'la floresta': 'Canelones',
    'tacuarembo' : 'Tacuarembo',
    'las delicias':'Paysandu',
    'paso de los toros': 'Tacuarembo',
    'sarandi del yi':'Durazno',
    'otra': 'Otra'
}

# Create the 'Departamentos' column by mapping the cleaned locality names
df['Departamentos'] = df['localidad_limpia'].map(departamentos_mapping)

# Fill any remaining NaN values (localities not in the mapping) with 'otros'
df['Departamentos'] = df['Departamentos'].fillna('otros')

# Display the value counts for the new 'Municipios' column
print(df['Departamentos'].value_counts())

In [ ]:
# reemplazar otros en departamentos si municipio distinto a otros
df.loc[(df['Departamentos'] == 'otros') & (df['Municipios'] != 'otros'), 'Departamentos'] = "Montevideo"

In [ ]:
print(df['Departamentos'].value_counts())

Unificamos departamento con municipios

In [ ]:
# muni_depto es municipios y si municpios=otros entonces departamento
df['muni_depto'] = df['Municipios']
df.loc[df['Municipios'] == 'otros', 'muni_depto'] = df['Departamentos']

In [ ]:
# Dejamos las que tienen localidad monteivode en el municipio mas frecuente (el B)
df.loc[df['muni_depto'] == 'Montevideo', 'muni_depto'] = "Municipio B"

In [ ]:
df['muni_depto'].value_counts()

In [ ]:
# Gráfico de barras con top 10 localidades
sns.countplot(x = 'muni_depto', data = df, color = 'blue',
              order = df['muni_depto'].value_counts().index);
plt.xticks(rotation=45, ha='right') # Rotate labels and align them to the right
plt.xlabel('muni_depto') # Change the x-axis label
plt.tight_layout() # Adjust layout to prevent labels from being cut off

**ORIGEN**

In [ ]:
df.ORIGEN.value_counts()

In [ ]:
sns.countplot(x = 'ORIGEN', data = df, color = 'blue',
              order = sorted(df['ORIGEN'].unique()));

**Observaciones:**

Como habíamos visto anteriormente la mayoría de los datos proviene del gallito

**FECHA**

In [ ]:
# Extraer el año y mes para graficar
df['FECHA_MONTH'] = df['FECHA'].dt.to_period('M')

# Contar publicaciones por mes
monthly_listings = df['FECHA_MONTH'].value_counts().sort_index()

# Convertir el PeriodIndex a strings para graficar
monthly_listings.index = monthly_listings.index.astype(str)

# Graficar el número de publicaciones a lo largo del tiempo
plt.figure(figsize=(12, 6))
monthly_listings.plot(kind='line')
plt.title('Número de Publicaciones por Mes según Fecha de Extracción')
plt.xlabel('Fecha de Extracción (Mes)')
plt.ylabel('Número de Publicaciones')
plt.xticks(rotation=45)
plt.grid(True)
plt.tight_layout()
plt.show()

**Observaciones:**

La fecha hace referencia al momento de la extracción de los datos (webscraping).
Como habíamos visto anteriormente los datos del Gallito son solo de 2023 y los de Mercado Libre se extienden desde 2023 hasta 2025, por lo tanto los datos en general se concentran en 2023.

## **Feature Engineering**

**Limpieza de descripción y título**

Generamos variables nuevas a partir de la descripción y el título (textos libres que por si mismos no aportan información al modelo)

In [ ]:
# --- 2. FUNCIÓN DE LIMPIEZA DE TEXTO ---
def limpiar_texto(texto):
    """
    Limpia un texto:
    - Maneja NaN (lo convierte en texto vacío).
    - Pasa a minúsculas.
    - Quita acentos.
    """
    if pd.isna(texto):
        return ""  # Devuelve un string vacío si es NaN

    # Pasa a minúsculas
    texto = str(texto).lower()

    # Quita acentos
    texto_normalizado = unicodedata.normalize('NFD', texto)
    texto_sin_acentos = "".join(
        [c for c in texto_normalizado if not unicodedata.combining(c)]
    )

    return texto_sin_acentos

# Aplicamos la limpieza a la columna de descripción
df['desc_limpia'] = df['DESCRIPCION'].apply(limpiar_texto)

# Aplicamos la limpieza a la columna de TITLE
df['title_limpia'] = df['TITLE'].apply(limpiar_texto)

# 2. ¡LIBERA MEMORIA! Borra la columna original, ya no la necesitas.
del df['DESCRIPCION']
del df['TITLE']

# 3. (Opcional) Fuerza al "recolector de basura" de Python a liberar RAM
import gc
gc.collect()

In [ ]:
df.head()

In [ ]:
# --- 3. CREACIÓN DE LAS NUEVAS VARIABLES (FEATURES) ---

# 3.1. Largo de la descripción
# .str.len() calcula la cantidad de caracteres.
# Si la descripción era NaN, ahora es "" (largo 0), lo cual es correcto.
df['largo_descripcion'] = df['desc_limpia'].str.len()


# 3.2. Definir las palabras clave (en minúscula y sin acentos)
# El pipe '|' actúa como un 'O' (OR)
keywords_amenities = (
    'piscina|pileta|piscinas|barbacoa|parrillero|gimnasio|gym|sum'
    '|salon|amenities|cancha|bbcoa|seguridad|bbq|vigilancia'
)

keywords_lujo = (
    'lujo|premium|alta gama|exclusivo|diseno|categoria|penthouse|jacuzzi'
)

keywords_vistas = (
    'vista al mar|frente al mar|primera linea|vista despejada'
    '|vistas|vista panoramica|mar|playa'
)

keywords_frente = (
    'frente|al frente'
)

keywords_contrafrente = (
    'contrafrente|contra frente|contra-frente'
)

keywords_interior = (
    'interior|interno'
)

keywords_lateral = (
    'lateral'
)

keywords_exterior = (
    'balcon|terraza|patio|tza'
)

keywords_nuevo = (
    'a estrenar|nuevo|estrena|estreno|nueva|moderno'
)

keywords_reciclado = (
    'reciclado'
)

keywords_en_pozo = (
    'en pozo|en construccion'
)

keywords_a_reciclar = (
    'a reciclar|para reciclar|reciclar'
)

keywords_piscina = (
    'piscina|pileta'
)

keywords_pozo_aire = (
    'pozo de aire'
)

keywords_garage = (
    'garage|garaje|gjes|gje|cochera'
)

In [ ]:
# 3.3. Crear las Dummies (Flags Binarias)
# .str.contains() devuelve True/False si encuentra alguna de las palabras.
# .astype(int) convierte True/False en 1/0, que es lo que el modelo necesita.

# Asegúrate de que las columnas limpias no tengan NaNs (sino strings vacíos)
# Si no lo hiciste antes, corre esto:
# df['desc_limpia'] = df['desc_limpia'].fillna("")
# df['title_limpia'] = df['title_limpia'].fillna("")

# 1. Amenities
df['dummy_amenities'] = (
    df['desc_limpia'].str.contains(keywords_amenities) |
    df['title_limpia'].str.contains(keywords_amenities)
).astype(int)

# 2. Lujo
df['dummy_lujo'] = (
    df['desc_limpia'].str.contains(keywords_lujo) |
    df['title_limpia'].str.contains(keywords_lujo)
).astype(int)

# 3. Vistas
df['dummy_vistas'] = (
    df['desc_limpia'].str.contains(keywords_vistas) |
    df['title_limpia'].str.contains(keywords_vistas)
).astype(int)

# 4. Frente
df['dummy_frente'] = (
    df['desc_limpia'].str.contains(keywords_frente) |
    df['title_limpia'].str.contains(keywords_frente)
).astype(int)

# 5. Contrafrente
df['dummy_contrafrente'] = (
    df['desc_limpia'].str.contains(keywords_contrafrente) |
    df['title_limpia'].str.contains(keywords_contrafrente)
).astype(int)

# 6. Exterior
df['dummy_exterior'] = (
    df['desc_limpia'].str.contains(keywords_exterior) |
    df['title_limpia'].str.contains(keywords_exterior)
).astype(int)

# 7. Nuevo
df['dummy_nuevo'] = (
    df['desc_limpia'].str.contains(keywords_nuevo) |
    df['title_limpia'].str.contains(keywords_nuevo)
).astype(int)

# 8. En Pozo
df['dummy_en_pozo'] = (
    df['desc_limpia'].str.contains(keywords_en_pozo) |
    df['title_limpia'].str.contains(keywords_en_pozo)
).astype(int)

# 9. Piscina
df['dummy_piscina'] = (
    df['desc_limpia'].str.contains(keywords_piscina) |
    df['title_limpia'].str.contains(keywords_piscina)
).astype(int)

# 10. Interior
df['dummy_interior'] = (
    df['desc_limpia'].str.contains(keywords_interior) |
    df['title_limpia'].str.contains(keywords_interior)
).astype(int)

# 11. Lateral
df['dummy_lateral'] = (
    df['desc_limpia'].str.contains(keywords_lateral) |
    df['title_limpia'].str.contains(keywords_lateral)
).astype(int)

# 12. Reciclado
df['dummy_reciclado'] = (
    df['desc_limpia'].str.contains(keywords_reciclado) |
    df['title_limpia'].str.contains(keywords_reciclado)
).astype(int)

# 13. A reciclar
df['dummy_a_reciclar'] = (
    df['desc_limpia'].str.contains(keywords_a_reciclar) |
    df['title_limpia'].str.contains(keywords_a_reciclar)
).astype(int)

# 14. Pozo de aire
df['dummy_pozo_aire'] = (
    df['desc_limpia'].str.contains(keywords_pozo_aire) |
    df['title_limpia'].str.contains(keywords_pozo_aire)
).astype(int)

# 15. Garage
df['dummy_garage'] = (
    df['desc_limpia'].str.contains(keywords_garage) |
    df['title_limpia'].str.contains(keywords_garage)
).astype(int)

In [ ]:
# 2. LIBERA MEMORIA. Borra la columna original, ya no la necesitas.
del df['desc_limpia']
del df['title_limpia']

# 3. (Opcional) Fuerza al "recolector de basura" de Python a liberar RAM
import gc
gc.collect()

In [ ]:
# describe solo con variables dummy
dummy_cols = [col for col in df.columns if col.startswith('dummy_')]
df[dummy_cols].describe().T['mean']

**Observaciones:**

- Verificamos que se generaron correctamente las dummies.
- El 33% de los apartamentos tiene amenities.
- El 25% de los apartamentos dan al frente.
- El 43% de los apartamentos tiene algun espacio exterior (balcón, terraza, patio).


In [ ]:
df.shape

**ITEM_CONDITION**

In [ ]:
df.ITEM_CONDITION.value_counts(dropna=False)

En este caso decidimos agrupar las categorías similares en una nueva variable.

In [ ]:
# quitamos espacios
df['ITEM_CONDITION'] = df['ITEM_CONDITION'].str.strip()

In [ ]:
# --- 2. DEFINIR EL DICCIONARIO DE MAPEO ---
# Basado en los grupos que definimos
mapeo_condicion = {
    # Grupo EN_OBRA
    'En construcción': 'EN_OBRA',
    'En pozo': 'EN_OBRA',

    # Grupo NUEVO
    'Estrenar': 'NUEVO',
    'Nuevo': 'NUEVO',

    # Grupo USADO_BUENO
    'Impecable': 'USADO_BUENO',
    'Buen estado': 'USADO_BUENO',
    'Reciclado': 'USADO_BUENO',

    # Grupo USADO_REGULAR
    'Usado': 'USADO_REGULAR',

    # Grupo A_RECICLAR
    'Para reciclar': 'A_RECICLAR',

    # Manejo de NaN
    np.nan: 'NO_ESPECIFICADO'
}

# --- 3. APLICAR EL MAPEO ---
# .map() aplica el diccionario a cada valor de la columna.
# Los valores que no están en el diccionario se convierten en NaN.
df['CONDITION_GROUP'] = df['ITEM_CONDITION'].map(mapeo_condicion)

# --- 4. VER EL RESULTADO ---
print("--- ITEM CONDITION Agrupado (Conteo) ---")
print(df['CONDITION_GROUP'].value_counts())

# Ahora, la columna 'CONDITION_GROUP' está lista para el modelo.

Completamos las dummies creadas con el valor de CONDITION GROUP

In [ ]:
# tabla cruzada de dummy_nuevo con condition_group=nuevo
pd.crosstab(df['CONDITION_GROUP'], df['dummy_nuevo'], dropna=False)

In [ ]:
# si condition group=NUEVO entonces dummy_nuevo=1
df.loc[df['CONDITION_GROUP'] == 'NUEVO', 'dummy_nuevo'] = 1

In [ ]:
# si condition group=EN_OBRA entonces dummy_en_pozo=1
df.loc[df['CONDITION_GROUP'] == 'EN_OBRA', 'dummy_en_pozo'] = 1

In [ ]:
# si condition group=A_RECICLAR entonces dummy_a_reciclar=1
df.loc[df['CONDITION_GROUP'] == 'A_RECICLAR', 'dummy_a_reciclar'] = 1

**ANTIGUEDAD**

Calculamos la antiguedad desde el año de construcción hasta el año de publicación.

In [ ]:
# calculamos antiguedad a partir de consctuction year y FECHA
df['ANTIGUEDAD'] = df['FECHA'].dt.year - df['CONSTRUCTION_YEAR']

In [ ]:
df['ANTIGUEDAD'].describe()

In [ ]:
# contamos casos con antiguedad negativa
(df['ANTIGUEDAD'] < 0).sum()

In [ ]:
# dejamos los negativo en 0
df.loc[df['ANTIGUEDAD'] < 0, 'ANTIGUEDAD'] = 0

In [ ]:
# describe de antiguedad para CONDITION_GROUP = NUEVO
df[df['CONDITION_GROUP'] == 'NUEVO']['ANTIGUEDAD'].describe()

En general tiene sentido (el 75% de los casos nuevos tienen antiguedad menor a 5 años) pero vemos valores incoherentes (máximo de 84 años). Consideramos que un apartamento es nuevo si tiene hasta 10 años de antiguedad.

In [ ]:
# COUNT DE MISSINGS DE ANTIGUEDAD
df['ANTIGUEDAD'].isna().sum()

# COUNT DE MISSINGS DE ANTIGUEDAD en porcentaje
df['ANTIGUEDAD'].isna().sum() / len(df)

**POSITION**

In [ ]:
df.POSITION.value_counts(dropna=False)

In [ ]:
# tabla cruzada position y origen con nulls
pd.crosstab(df['POSITION'], df['ORIGEN'], dropna=False)

En este caso, POSITION es una variable de Gallito pero que viene vacía en muchos casos.

Completamos las dummies creadas con el valor de POSITION

In [ ]:
# quitar esparcio a position
df['POSITION'] = df['POSITION'].str.strip()

In [ ]:
# Si POSITION=Frente entonces dummy_frente=1
df.loc[df['POSITION'] == 'Frente', 'dummy_frente'] = 1

In [ ]:
# Si POSITION=Contrafrente entonces dummy_contra_frente=1
df.loc[df['POSITION'] == 'Contrafrente', 'dummy_contrafrente'] = 1

In [ ]:
# Si POSITION=Interior entonces dummy_interior=1
df.loc[df['POSITION'] == 'Interior', 'dummy_interior'] = 1

In [ ]:
# Si POSITION=Lateral entonces dummy_lateral=1
df.loc[df['POSITION'] == 'Lateral', 'dummy_lateral'] = 1

**RATIO BAÑOS/ DORMITORIOS**

In [ ]:
# Reemplazamos 0s en 'BEDROOMS' con 1, solo para esta operación.
# Esto evita la división por cero.
denominador = df['BEDROOMS'].replace(0, 1)

# Ahora podemos dividir de forma segura
df['RATIO_BATH_BED'] = df['FULL_BATHROOMS'] / denominador

In [ ]:
df['RATIO_BATH_BED'].describe()

In [ ]:
# Vemos percentiles de RATIO_BATH_BED
df['RATIO_BATH_BED'].describe(percentiles=[0.01, 0.015, 0.05,0.075,  0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.975, 0.99, 0.995])

In [ ]:
# dejamos en 1 los valores mayores a 1 de RATIO_BATH_BED
df.loc[df['RATIO_BATH_BED'] > 1, 'RATIO_BATH_BED'] = 1

In [ ]:
# dejamos en 2 decimales
df['RATIO_BATH_BED'] = df['RATIO_BATH_BED'].round(2)

In [ ]:
# Gráfico de barras
sns.countplot(x = 'RATIO_BATH_BED', data = df, color = 'blue');
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

**Compeltamos dummies**

In [ ]:
# si garage>0 entonces dummy_garage=1
df.loc[df['GARAGE'] > 0, 'dummy_garage'] = 1

In [ ]:
df.dummy_garage.value_counts()

## Análisis Bivariado

A continuación generamos gráficos para analizar al relación entre cada una de las variables independientes y nuestra variable objetivo (precio)

**Variables continuas**

In [ ]:
fig, axes = plt.subplots(1, 2, figsize = (20, 6))

fig.suptitle("Gráficos de dispersión para Área y Antiguedad con respecto al Precio")
sns.scatterplot(x = 'COVERED_AREA', y = 'PRICE', data = df, ax = axes[0]);
sns.scatterplot(x = 'ANTIGUEDAD', y = 'PRICE', data = df, ax = axes[1]);


**Observaciones de los gráficos de dispersión:**

- **Área Cubierta (COVERED_AREA) vs. Precio (PRICE):** Se observa una relación positiva. A medida que aumenta el área cubierta, el precio de los apartamentos tiende a ser mayor. La concentración de puntos en la parte inferior izquierda indica que la mayoría de los apartamentos son más pequeños y menos costosos.
- **Antigüedad (ANTIGUEDAD) vs. Precio (PRICE):** Se aprecia una relación inversa. Los apartamentos con menor antigüedad (más recientes) tienden a tener precios más altos, mientras que los apartamentos con mayor antigüedad generalmente tienen precios más bajos.

**LOCALIDADES**

In [ ]:
# Calculate the order based on the average price
price_order = df.groupby('muni_depto')['PRICE'].mean().sort_values()

fig, ax1 = plt.subplots(figsize=(18, 6))

# Bar plot for count of apartments by muni_depto on the primary y-axis (ax1)
sns.countplot(x='muni_depto', data=df, ax=ax1, color='blue', order=price_order.index) # Use index for order
ax1.set_title('Frecuencia y precio promedio por Municipio/Departamento') # Updated title
ax1.set_xlabel('Municipio/Departmento') # Updated xlabel
ax1.set_ylabel('Count', color='blue')
ax1.tick_params(axis='y', labelcolor='blue')
plt.xticks(rotation=45, ha='right') # Rotate x-axis labels

# Create a secondary y-axis for the line plot
ax2 = ax1.twinx()

# Line plot for average price by muni_depto on the secondary y-axis (ax2)
# Plotting the line using the ordered index and values from price_order
sns.lineplot(x=price_order.index, y=price_order.values, ax=ax2, marker='o', color='red')
ax2.set_ylabel('Precio Promedio', color='red')
ax2.tick_params(axis='y', labelcolor='red')

plt.tight_layout()
plt.show()

In [ ]:
# Filter out the 'Otra', 'otros', and 'Otros' categories from the DataFrame before plotting
df_filtered = df[~df['muni_depto'].isin(['Otra', 'otros', 'Otros'])]

# Calculate the order based on the median price
median_price_order = df_filtered.groupby('muni_depto')['PRICE'].median().sort_values(ascending=False).index

# box plot de PRICE por muni_depto
plt.figure(figsize = (20, 10))
sns.boxplot(x='muni_depto', y='PRICE', data=df_filtered, order=median_price_order) # Added order
plt.ylabel('Precio')
plt.xlabel('Municipio/Departamento')
plt.xticks(rotation=45, ha='right') # Rotate labels for readability
plt.tight_layout() # Adjust layout
plt.show()

**Observaciones:**

- Hay diferencias significativas en los precios entre los distintos municipios/departamentos, con Maldonado, Municipio E, y Municipio CH presentando generalmente precios más altos, y Municipio F, Municipio A, y Municipio G con precios más bajos.
- La dispersión de los precios también varía entre los grupos, indicando que algunos municipios/departamentos tienen una mayor variabilidad en los precios de los apartamentos que otros.
- Los puntos individuales más allá de los bigotes representan posibles valores atípicos dentro de cada grupo.

**FULL_BATHROOMS**

In [ ]:
fig, ax1 = plt.subplots(figsize=(18, 6))

# Bar plot for count of apartments by number of full bathrooms on the primary y-axis (ax1)
sns.countplot(x='FULL_BATHROOMS', data=df, ax=ax1, color='blue')
ax1.set_title('Frecuencia y precio promedio por Baños')
ax1.set_xlabel('Cantidad de baños completos')
ax1.set_ylabel('Count', color='blue')
ax1.tick_params(axis='y', labelcolor='blue')

# Create a secondary y-axis for the line plot
ax2 = ax1.twinx()

# Line plot for average price by number of full bathrooms on the secondary y-axis (ax2)
sns.lineplot(x='FULL_BATHROOMS', y='PRICE', data=df, ax=ax2, marker='o', color='red', ci=None)
ax2.set_ylabel('Precio Promedio', color='red')
ax2.tick_params(axis='y', labelcolor='red')

plt.tight_layout()
plt.show()

In [ ]:
# box plot de PRICE por FULL_BATHROOMS
plt.figure(figsize = (20, 10))
sns.boxplot(x='FULL_BATHROOMS', y='PRICE', data=df)
plt.ylabel('PRICE')
plt.xlabel('FULL_BATHROOMS')
plt.show()

**ITEM_CONDITION**

In [ ]:
# TABLA CRUZADA DE ITEM CONDITION Y CONDITION_GROUP
pd.crosstab(df['CONDITION_GROUP'], df['ITEM_CONDITION'], dropna=False)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Calcular el orden basado en el precio promedio (esto sigue igual)
price_order = df.groupby('ITEM_CONDITION')['PRICE'].mean().sort_values().index

# 2. Crear los gráficos
fig, ax1 = plt.subplots(figsize=(18, 6))

# Gráfico de barras (Conteo) en el eje primario (ax1) - Con orden
sns.countplot(
    x='ITEM_CONDITION',
    data=df,
    ax=ax1,
    color='blue',
    order=price_order  # Correcto: countplot sí acepta 'order'
)
ax1.set_title('Conteo y Precio Promedio por Condición del Apartamento (Ordenado por Precio)')
ax1.set_xlabel('Condición del Apartamento')
ax1.set_ylabel('Conteo', color='blue')
ax1.tick_params(axis='y', labelcolor='blue')
ax1.set_xticklabels(ax1.get_xticklabels(), rotation=45, ha='right')

# Eje secundario (ax2)
ax2 = ax1.twinx()

# Gráfico de línea (Promedio) en el eje secundario (ax2) - Sin orden
sns.lineplot(
    x='ITEM_CONDITION',
    y='PRICE',
    data=df,
    ax=ax2,
    marker='o',
    color='red',
    errorbar=None  # Corregido: Reemplaza a ci=None
    # Se elimina la línea 'order=price_order'
)
ax2.set_ylabel('Precio Promedio', color='red')
ax2.tick_params(axis='y', labelcolor='red')

plt.tight_layout()
plt.show()

**BEDROOMS**

In [ ]:
plt.figure(figsize = (20, 10))
sns.boxplot(x='BEDROOMS', y='PRICE', data=df)
plt.ylabel('PRICE')
plt.xlabel('BEDROOMS')
plt.show()

**GARAGE**

In [ ]:
plt.figure(figsize = (20, 10))
sns.boxplot(x='GARAGE', y='PRICE', data=df)
plt.ylabel('PRICE')
plt.xlabel('GARAGE')
plt.show()

**ORIGEN**

In [ ]:
plt.figure(figsize = (10, 5))
sns.boxplot(x='ORIGEN', y='PRICE', data=df)
plt.ylabel('PRICE')
plt.xlabel('ORIGEN')
plt.show()

**POSITION**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# 1. Calcular el orden basado en el precio promedio (esto sigue igual)
price_order = df.groupby('POSITION')['PRICE'].mean().sort_values().index

# 2. Crear los gráficos
fig, ax1 = plt.subplots(figsize=(18, 6))

# Gráfico de barras (Conteo) en el eje primario (ax1) - Con orden
sns.countplot(
    x='POSITION',
    data=df,
    ax=ax1,
    color='blue',
    order=price_order  # Correcto: countplot sí acepta 'order'
)
ax1.set_title('Conteo y Precio Promedio por POSITION del Apartamento (Ordenado por Precio)')
ax1.set_xlabel('POSITION del Apartamento')
ax1.set_ylabel('Conteo', color='blue')
ax1.tick_params(axis='y', labelcolor='blue')
ax1.set_xticklabels(ax1.get_xticklabels(), rotation=45, ha='right')

# Eje secundario (ax2)
ax2 = ax1.twinx()

# Gráfico de línea (Promedio) en el eje secundario (ax2) - Sin orden
sns.lineplot(
    x='POSITION',
    y='PRICE',
    data=df,
    ax=ax2,
    marker='o',
    color='red',
    errorbar=None  # Corregido: Reemplaza a ci=None
    # Se elimina la línea 'order=price_order'
)
ax2.set_ylabel('Precio Promedio', color='red')
ax2.tick_params(axis='y', labelcolor='red')

plt.tight_layout()
plt.show()

**Dummies**

In [ ]:
dummy_cols = ['dummy_amenities', 'dummy_lujo', 'dummy_vistas', 'dummy_frente',
              'dummy_contrafrente', 'dummy_exterior', 'dummy_nuevo', 'dummy_en_pozo',
              'dummy_piscina', 'dummy_interior', 'dummy_lateral', 'dummy_reciclado',
              'dummy_a_reciclar', 'dummy_pozo_aire', 'dummy_garage']

# Calculate the number of rows and columns for subplots
n_cols = 3
n_rows = (len(dummy_cols) + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(18, n_rows * 5))
axes = axes.flatten() # Flatten the 2D array of axes for easy iteration

for i, col in enumerate(dummy_cols):
    sns.boxplot(x=col, y='PRICE', data=df, ax=axes[i])
    axes[i].set_title(col)
    axes[i].set_ylabel('PRICE')
    axes[i].set_xlabel(col)

# Hide any unused subplots
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

**Observaciones:**

- La mayoría de los boxplots muestran diferencias en los precios entre los apartamentos que poseen la característica y los que no


- Posiciones como contrafrente, lateral e interior tienen precios más bajos en promedio


- Apartamentos con pozo de aire, a reciclar o en pozo, también suelen ser más baratos


- Aquellos con garage y/o piscina son notoriamente más caros que los que no tienen


## **Análisis Multivariado**

### **Matriz de Correlaciones**

In [ ]:
fig = plt.figure(figsize = (18, 6))

sns.heatmap(df.corr(numeric_only = True), annot = True);

plt.xticks(rotation = 45);

**Matriz de correlacion con las variables correlacionadas > 0.4**

In [ ]:
# Calculate the correlation matrix
correlation_matrix = df.corr(numeric_only=True)

# Filter the correlation matrix for values greater than 0.4 (absolute value), excluding the diagonal
filtered_correlation_matrix = correlation_matrix[abs(correlation_matrix) > 0.4]
np.fill_diagonal(filtered_correlation_matrix.values, np.nan) # Set diagonal to NaN to exclude

# Display the filtered correlation matrix, potentially using a heatmap
plt.figure(figsize=(12, 8))
sns.heatmap(filtered_correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=.5)
plt.title('Correlation Matrix (Absolute Correlation > 0.4)')
plt.show()

**Observaciones:**

- Correlaciones fuertes con precio

Área cubierta (0.73)

Cantidad de baños (071)

Cantidad de dormitorios (0.47)

- Correlaciones entre variables independientes

Área cubierta con cantidad de baños (0.73)

Cantidad de dormitorios con cantidad de baños (0.59)

Área cubierta con garage (0.67)


In [ ]:
import os

# Definimos la ruta del directorio
output_dir = '/content/drive/MyDrive/projecto uruguay'

# Creamos el directorio
os.makedirs(output_dir, exist_ok=True)

# Guardamos base final (mas liviana)
df.to_excel(os.path.join(output_dir, 'ventas_aptos_final.v.xlsx'), index=False)

# **4- Preparación de los datos para el modelo**

* Antes de proceder a construir un modelo, tendremos que codificar las características categóricas.
* Separar las variables independientes y las variables dependientes.
* Dividiremos los datos en entrenamiento y prueba para poder evaluar el modelo que entrenamos con los datos de entrenamiento.

In [ ]:
# leemos la base final (post tratamiento)
#df = pd.read_excel('/content/drive/MyDrive/projecto uruguay/ventas_aptos_final.v.xlsx')
df = pd.read_excel('/content/drive/MyDrive/Proyecto Aptos/ventas_aptos_final.v.xlsx')


In [ ]:
# Generamos una copia para no modificar la base original
datos = df.copy()

In [ ]:
df.head()

In [ ]:
# Revisamos el porcentaje de missings por variable
df.isnull().sum()*100/len(df)

* Vamos a eliminar las variables que quedaron con missings luego del tratamiento.
* Notamos que el año de construcción podría aportar valor a la dummy nuevo.

In [ ]:
# table de dummy_nuevo y antiguedad<10
pd.crosstab(df['dummy_nuevo'], df['ANTIGUEDAD'])

Consideramos que un apartamento es nuevo si tiene una antiguedad menor a 5 años

In [ ]:
# si ANTIGUEDAD< 5 entonces dummy_nuevo=1
df.loc[df['ANTIGUEDAD'] < 5, 'dummy_nuevo'] = 1

In [ ]:
df['dummy_nuevo'].value_counts()

**Observaciones:**

Un alto porcentaje de los apartamentos son nuevos. Tiene coherencia con la situación del mercado actual.

In [ ]:
# guardamos el df para usar mas adelante en Cat boost
df_cat=df

### Eliminación de variables con missings

Las demás variables se eliminan. En la etapa anterior generamos dummies para sustituirlas.

In [ ]:
# eliminamos año de construccion, antiguedad, position, condition
df = df.drop(['CONSTRUCTION_YEAR', 'ANTIGUEDAD', 'POSITION', 'ITEM_CONDITION', 'CONDITION_GROUP'], axis = 1)

In [ ]:
# eliminamos ORIGEN, ya que luego del análisis bivariado no la consideramos relevante para el modelo
df = df.drop(['ORIGEN'], axis = 1)

In [ ]:
# eliminamos otras variables auxiliares: fecha, fecha_month, localidad, localidad_limpia, municipios, departamentos, largo_descripcion
df = df.drop(['FECHA', 'FECHA_MONTH', 'LOCALIDAD', 'localidad_limpia', 'Municipios', 'Departamentos', 'largo_descripcion', "GARAGE"], axis = 1)

In [ ]:
df.head()

Creamos variables dummy para las variables categóricas

In [ ]:
df = pd.get_dummies(
    df,
    columns = df.select_dtypes(include = ["object", "category"]).columns.tolist(),
    drop_first = True,
)

In [ ]:
df.head()

In [ ]:
# Separamos variables independientes de la variable target
x = df.drop('PRICE',axis=1)

y = df['PRICE']

In [ ]:
# Separamos el dataset en train y test
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.2, shuffle = True, random_state = 1)

In [ ]:
# Revisamos las dimesiones
print("Shape of Training set : ", x_train.shape)
print("Shape of test set : ", x_test.shape)

# **5- Desarrollo del modelo**

**Función para generar medidas de performance**

Para evaluar la calidad de este modelo de regresión, utilizaremos un conjunto estándar de métricas.

El Error Cuadrático Medio (RMSE) y el Error Absoluto Medio (MAE) cuantifican la magnitud promedio de los errores de predicción, siendo el RMSE más sensible a errores grandes.

El Coeficiente de Determinación (R2) indica la proporción de la varianza en la variable dependiente que es explicada por el modelo; un valor cercano a 1 sugiere un buen ajuste. El R2 Ajustado es una versión mejorada que penaliza la inclusión de variables irrelevantes, siendo crucial para comparar modelos con diferente número de predictores.

Finalmente, el Error Porcentual Absoluto Medio (MAPE) expresa el error promedio como un porcentaje de los valores reales, lo que facilita la interpretación del rendimiento en términos relativos.

In [ ]:
# Function to compute adjusted R-squared
def adj_r2_score(predictors, targets, predictions):
    r2 = r2_score(targets, predictions)
    n = predictors.shape[0]
    k = predictors.shape[1]
    return 1 - ((1 - r2) * (n - 1) / (n - k - 1))


# Function to compute MAPE
def mape_score(targets, predictions):
    return np.mean(np.abs(targets - predictions) / targets) * 100


# Function to compute different metrics to check performance of a regression model
def model_performance_regression(model, predictors, target):
    """
    Function to compute different metrics to check regression model performance

    model: regressor
    predictors: independent variables
    target: dependent variable
    """

    pred = model.predict(predictors)                  # Predict using the independent variables
    r2 = r2_score(target, pred)                       # To compute R-squared
    adjr2 = adj_r2_score(predictors, target, pred)    # To compute adjusted R-squared
    rmse = np.sqrt(mean_squared_error(target, pred))  # To compute RMSE
    mae = mean_absolute_error(target, pred)           # To compute MAE
    mape = mape_score(target, pred)                   # To compute MAPE

    # Creating a dataframe of metrics
    df_perf = pd.DataFrame(
        {
            "RMSE": rmse,
            "MAE": mae,
            "R-squared": r2,
            "Adj. R-squared": adjr2,
            "MAPE": mape,
        },
        index=[0],
    )

    return df_perf

## Modelo I: Árbol de decisión

Entrenamos el modelo con la muestra de train y revisamos su performance en la muestra de test

In [ ]:
# Decision Tree Regressor
dt_regressor = DecisionTreeRegressor(random_state = 1)

# Fitting the model
dt_regressor.fit(x_train, y_train)

# Model Performance on the test data, i.e., prediction
dt_regressor_perf_test = model_performance_regression(dt_regressor, x_test, y_test)

dt_regressor_perf_test

Este modelo explica bien los precios:
- captura el 72% de la varianza en los datos
- su error absoluto promedio es de USD 30,582
- tiene una desviación porcentual promedio del 17.19%.

**Graficamos el árbol**

In [ ]:
from sklearn import tree
features = list(x.columns)

# Building the model with max_depth=3
dt_regressor_visualize = DecisionTreeRegressor(random_state = 1, max_depth=3)

# Fitting the model
dt_regressor_visualize.fit(x_train, y_train)


plt.figure(figsize = (20, 20))
tree.plot_tree(dt_regressor_visualize, feature_names = features, filled = True, fontsize = 12,
               node_ids = True, class_names = True)
plt.show()

In [ ]:
print(tree.export_text(dt_regressor_visualize, feature_names=x_train.columns.tolist(), show_weights=True))

- **Baños y Área**: Son los principales impulsores positivos del precio.

- **Ubicación (Municipio B)**: Es un factor de descuento significativo que anula las ganancias de precio obtenidas por tener grandes áreas y varios baños.

- **Garage**: Es un factor de mejora de precio, pero solo para los apartamentos más pequeños y con menos baños.

In [ ]:
# Plotting the feature importance
features = list(x_train.columns)

# Get feature importances from the trained model
importances = dt_regressor.feature_importances_

indices = np.argsort(importances)

plt.figure(figsize = (10, 10))

plt.title('Feature Importances')

plt.barh(range(len(indices)), importances[indices], color = 'blue', align = 'center')

plt.yticks(range(len(indices)), [features[i] for i in indices])

plt.xlabel('Relative Importance')

plt.show()

Como vimos en el árbol, las variables considerablemente más importantes son: FULL_BATHROOMS, COVERED_AREA y muni_depto_Municipio_B. Le siguen las dummies de garage y nuevo.

## Modelo II: Random Forest

In [ ]:
# Random Forest Regressor
rf_regressor = RandomForestRegressor(n_estimators = 100, random_state = 1)

# Fitting the model
rf_regressor.fit(x_train, y_train)

# Model Performance on the test data
rf_regressor_perf_test = model_performance_regression(rf_regressor, x_test, y_test)

rf_regressor_perf_test

El error porcentual promedio se redujo en 1 punto porcentual. En promedio, la predicción del modelo se desvía del precio real en solo un 16.08%.

El Random Forest es claramente mejor que el árbol de decisión:

- **Mayor Precisión:** Logra reducir el error absoluto promedio (MAE) en casi USD 2,400, lo que se traduce en un error promedio porcentual (MAPE) más bajo.

- **Mejor Ajuste:** Su capacidad para explicar la varianza en los precios (R2) subió del 72% al 79%, indicando una comprensión mucho más sólida de los factores que influyen en el precio.

Es esperable que el random forest sea más preciso que un árbol ya que soluciona los 2 grandes problemas de los árboles (alta variabilidad y sobreajuste). El random forest tiene doble aleatorización: genera muchos arboles con distintas muestras y además muestrea las variables. Esto logra que las predicciones sean más estables (promedio de todos los arboles) y más generales (se aprenden patrones generales).

## Modelo III: AdaBoost

In [ ]:
# Importing AdaBoost Regressor
from sklearn.ensemble import AdaBoostRegressor

# AdaBoost Regressor
ada_regressor = AdaBoostRegressor(random_state=1)

# Fitting the model
ada_regressor.fit(x_train, y_train)

# Model Performance on the test data
ada_regressor_perf_test = model_performance_regression(ada_regressor, x_test, y_test)

ada_regressor_perf_test

El modelo AdaBoost tiene un rendimiento deficiente en comparación con los modelos de Random Forest y Árbol de Decisión simple para este problema.

- Bajo Poder Explicativo: Solo explica el 62% de la varianza.

- Alto Error: El error promedio (MAE) es de más de USD 44k y el error porcentual (MAPE) está cerca del 30%.

AdaBoost es muy sensible al ruido y a los outliers. Si los datos de precios de apartamentos contienen muchos valores atípicos, AdaBoost podría estar forzando a los árboles a enfocarse en esos casos difíciles en cada iteración, lo que resulta en un modelo final sobreajustado al ruido y con peor rendimiento general en el conjunto de prueba.

## Modelo IV: XGBoost

In [ ]:
# Installing the xgboost library using the 'pip' command
!pip install xgboost

In [ ]:
# Importing XGBoost Regressor
from xgboost import XGBRegressor

# XGBoost Regressor
xgb = XGBRegressor(random_state = 1)

# Fitting the model
xgb.fit(x_train,y_train)

# Model Performance on the test data
xgb_perf_test = model_performance_regression(xgb, x_test, y_test)

xgb_perf_test

Este es un modelo sólido con mejor rendimiento que el árbol de decisión y que el AdaBoost. Sin embargo, el random forest sigue siendo el modelo con el mejor rendimiento general debido a su mayor R2 (0.76) y sus menores errores (MAE de 28.2k y MAPE de 16.08%).

## Modelo V: CAT Boost

In [ ]:
!pip install catboost

In [ ]:
from catboost import CatBoostRegressor, Pool, cv

### Tratamiento de variables para CAT

Para CAT Boost no eliminamos variables solo por tener missings, ya que maneja los valores nulos de forma nativa. Aprenderá si el hecho de que no esté especificado es en sí mismo un factor que afecta el precio. En este caso mantenemos antiguedad y position.

In [ ]:
# eliminamos año de construccion, condition
df_cat = df_cat.drop(['CONSTRUCTION_YEAR', 'ITEM_CONDITION', 'CONDITION_GROUP'], axis = 1)

In [ ]:
# eliminamos ORIGEN, ya que luego del análisis bivariado no la consideramos relevante para el modelo
df_cat = df_cat.drop(['ORIGEN'], axis = 1)

In [ ]:
# eliminamos otras variables auxiliares: fecha, fecha_month, localidad, localidad_limpia, municipios, departamentos, largo_descripcion
df_cat = df_cat.drop(['FECHA', 'FECHA_MONTH', 'LOCALIDAD', 'localidad_limpia',  'Municipios', 'Departamentos', 'largo_descripcion', "GARAGE"], axis = 1)

In [ ]:
df_cat.info()

In [ ]:
# Asignamos -1 a los missings de antiguedad para que CatBoost aprenda que es una categoria especial
df_cat['ANTIGUEDAD'] = df_cat['ANTIGUEDAD'].fillna(-1)

In [ ]:
df_cat['POSITION'].value_counts()

In [ ]:
# Asignamos no_especificado a los missings de position
df_cat['POSITION'] = df_cat['POSITION'].fillna('no_especificado')

#### **1er paso.** Identificar la variable objetivo y ver su proporción

In [ ]:
# Separating independent variables and the target variable
x = df_cat.drop('PRICE',axis=1)

y = df_cat['PRICE']

In [ ]:
# Splitting the dataset into train and test datasets
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.2, shuffle = True, random_state = 1)

In [ ]:
# Checking the shape of the train and test data
print("Shape of Training set : ", x_train.shape)
print("Shape of test set : ", x_test.shape)

#### Identificar las variables categóricas, dado que catboost las trata de forma diferencial

In [ ]:
x.info()

In [ ]:
cat_features = x.select_dtypes(include=["object"]).columns.tolist()
cat_features

#### **2do.** Corroboramos que las no categóricas son continuas

In [ ]:
no_cat_features = x.select_dtypes(exclude=["object"]).columns.tolist()
x[no_cat_features].mean()

En catboost la gestión de los datos se da mediante el objeto "Pool". Este optimiza el manejo de las variables categóricas y gestión de nan

In [ ]:
train_pool = Pool(x_train, y_train, cat_features=cat_features)
test_pool = Pool(x_test, y_test, cat_features=cat_features)

#### 3ero. Definimos el modelo

In [ ]:
# 3. Definir modelo CatBoostRegressor
model = CatBoostRegressor(
    iterations=200,          # Número máximo de árboles (en 3.000 demora 20 minutos)
    learning_rate=0.05,       # Tasa de aprendizaje
    depth=8,                  # Profundidad de los árboles
    loss_function='RMSE',     # Función de pérdida para regresión (default)
    eval_metric='MAPE',       # Métrica de seguimiento para regresión
    random_seed=42,
    # DETECCIÓN DE SOBREAJUSTE (OVERFITTING DETECTOR)
    # 1. od_type='IncToDec': Activa el detector que busca la disminución del rendimiento.
    #    El entrenamiento se detiene si la métrica de la validación deja de mejorar
    #    (o empieza a empeorar) durante 'od_wait' rondas.
    od_type='IncToDec',

    # 2. od_wait: Paciencia del detector. Rondas sin mejora antes de detenerse.
    od_wait=50,
    verbose=200
)

#### 4to. Aplicar cross validation para estudiar las métricas del modelo

In [ ]:
cv_results = cv(
    params=model.get_params(),
    pool=train_pool,
    fold_count=5,
    shuffle=True,
    partition_random_seed=42,
    verbose=False
)

# 1. La métrica ahora es 'RMSE' (o la que pusiste en eval_metric)
# 2. Buscamos el MÍNIMO error, no el máximo
print("Best (min) RMSE en CV:", min(cv_results["test-RMSE-mean"]))

# También es muy común ver el valor en la última iteración (después del early stopping)
print("RMSE en la última iteración:", cv_results["test-RMSE-mean"].iloc[-1])

In [ ]:
# Asumiendo que 'model' ya es un CatBoostRegressor
model.fit(train_pool, eval_set=test_pool, use_best_model=True)

y_pred = model.predict(test_pool)

In [ ]:
# Model Performance on the test data, i.e., prediction
dt_catboost_perf_test = model_performance_regression(model, x_test, y_test)

dt_catboost_perf_test

## Comparativo de modelos

In [ ]:
models_test_comp_df = pd.concat(
    [
        dt_regressor_perf_test.T,
        rf_regressor_perf_test.T,
        ada_regressor_perf_test.T,
        xgb_perf_test.T,
        dt_catboost_perf_test.T
    ],
    axis = 1,
)

models_test_comp_df.columns = [
    "Decision tree regressor",
    "Random Forest regressor",
    "Ada Boost Regressor",
    "XG Boost Regressor",
    "CatBoost Regressor"
]

print("Test performance comparison:")

models_test_comp_df.T

El mejor rendimiento lo obtuvo el Random Forest Regressor. Este modelo presenta el menor Error Cuadrático Medio (RMSE) con 43194.92, el menor Error Absoluto Medio (MAE) con 28180.64, y el menor Error Porcentual Absoluto Medio (MAPE) con 16.08%. Además, es el que mejor explica la varianza de los datos, con un R-squared y R-squared Ajustado del 0.79. Esto indica que es el modelo más preciso y robusto para este problema.


Si bien hicimos distintas pruebas con otros modelos (en particular catboost el modelo que mas competia), pero ninguno de esta se aproxima al buen resultado de Random Forest.

Por lo cual, seleccionamos el Random Forest Regressor para cumplir con el objetivo propuesto.

Una vez seleccionado el modelo Random Forest como el de mejor rendimiento inicial, procederemos a la optimización de sus hiperparámetros con el objetivo de mejorar aún más su precisión y robustez.



## Optimización de hiperparametros con optuna

In [ ]:
def objective(trial):
    # 1. Definición de Hiperparámetros (Espacio de Búsqueda Acotado)
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 300, step=50),
        'max_depth': trial.suggest_int('max_depth', 10, 30, step=5),
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 5, step=1),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2', 0.3, 0.5]),
        'random_state': 42,
        'n_jobs': -1  # Usa todos los cores de tu CPU para acelerar
    }

    # 2. Configuración de Validación Cruzada Simplificada
    # Usamos 3 folds para mayor rapidez.
    kf = KFold(n_splits=3, shuffle=True, random_state=42)
    mape_scores = []

    # Iterar sobre los folds de CV
    for fold, (train_index, val_index) in enumerate(kf.split(x_train)):
        # Si usas submuestreo, aplica .iloc[] o .sample() aquí
        X_tr, X_val = x_train.iloc[train_index], x_train.iloc[val_index]
        y_tr, y_val = y_train.iloc[train_index], y_train.iloc[val_index]

        # Entrenamiento del modelo
        model = RandomForestRegressor(**params)
        model.fit(X_tr, y_tr)

        # Predicción y Cálculo de MAPE
        preds = model.predict(X_val)
        current_mape = mape(y_val, preds)
        mape_scores.append(current_mape)

    # Optuna busca MINIMIZAR el valor retornado (el MAPE promedio)
    return np.mean(mape_scores)

In [ ]:
# 3. Creación y Ejecución del Estudio de Optuna

# Asegúrate de haber definido X_train y y_train antes de esto
study = optuna.create_study(direction='minimize')
study.optimize(objective, n_trials=50, show_progress_bar=True)

# Imprimir los mejores resultados
print(f"Mejor MAPE encontrado: {study.best_value:.4f}")
print("Mejores hiperparámetros:")
print(study.best_params)

- La optimización final demoró 54 minutos
- El mejor MAPE encontrado fue de 0.1641
- Los mejores hiperparámetros fueron: {'n_estimators': 250, 'max_depth': 25, 'min_samples_split': 2, 'max_features': 0.3}
- Luego de varias pruebas, decidimos quedarnos con esta combinación de hiperparámetros. Ya que nos pareció la más eficiente en términos de costo computacional, tiempo y resultados finales

**Hiperparámetros finales:**

- n_estimators: es el número de árboles del random forest, a mayor cantidad, más preciso es el modelo pero también es más costoso computacionalmente.
- max_depth: es la profundidad máxima que puede tener cada árbol. Limitarla ayuda a evitar el sobreajuste.
- min_sample_split: es el mínimo de muestras para la división de un nodo interno. Aumentar este valor puede regularizar el modelo y prevenir que los árboles se ajusten demasiado a los datos de entrenamiento.
- max_features: es el porcentaje de características a considerar al buscar la mejor división en un nodo. Random Forest selecciona un subconjunto aleatorio de características para cada división, lo que introduce aleatoriedad y hace que los árboles sean más diversos.

Generamos nuevamente las mismas bases de train y test utilizadas en el Random Forest incial (random_state=1)

In [ ]:
# Separamos variables independientes de la variable target
x = df.drop('PRICE',axis=1)

y = df['PRICE']

In [ ]:
# Separamos el dataset en train y test
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.2, shuffle = True, random_state = 1)

In [ ]:
# Revisamos las dimesiones
print("Shape of Training set : ", x_train.shape)
print("Shape of test set : ", x_test.shape)

In [ ]:
# Probamos los parametros que da optuna sobre el modelo de random forest que corrimos inicialmente (con la base de test)

# Random Forest Regressor
rf_regressor_op = RandomForestRegressor (n_estimators = 250, max_depth = 25, min_samples_split = 2, max_features = 0.3, random_state = 1)

# Fitting the model
rf_regressor_op.fit(x_train, y_train)

# Model Performance on the test data
rf_regressor_perf_test = model_performance_regression(rf_regressor_op, x_test, y_test)

rf_regressor_perf_test

Notamos una mejora con respecto al modelo original:
- El MAPE disminuye de 16.08 a 15.90
- El MAE disminuye de 28,180.64 a 27,861.47
- El R cuadrado aumenta de 0.79 a 0.80

In [ ]:
# y_pred_rf se obtiene de la predicción del modelo rf_regressor_op sobre x_test.
y_pred_rf = rf_regressor_op.predict(x_test)

## Gráfico de Feature Importance

In [ ]:
# Plotting the feature importance
features = list(x_train.columns)

# Get feature importances from the trained model
importances = rf_regressor_op.feature_importances_

indices = np.argsort(importances)

plt.figure(figsize = (10, 10))

plt.title('Feature Importances')

plt.barh(range(len(indices)), importances[indices], color = 'violet', align = 'center')

plt.yticks(range(len(indices)), [features[i] for i in indices])

plt.xlabel('Relative Importance')

plt.xlim(right=0.5)

plt.show()

Variables dominantes:

- **COVERED_AREA**: Es la característica más importante, con una importancia relativa superior a 0.35. Esto indica que el modelo utiliza la superficie del inmueble como el factor predictor principal de su valor.

- **FULL_BATHROOMS**: Es la segunda más importante, aunque su peso es significativamente menor que la primera (alrededor de 0.12).

- **BEDROOMS**: Es la tercera, con una importancia apenas por debajo de FULL_BATHROOMS.

Otras variables relevantes:

**muni_depto_Municipio B**, **RATIO_BATH_BED**, **muni_depto_Maldonado** son importantes, pero con un peso mucho menor, todas con valores por debajo de 0.05.

## Gráfico de Dispersión: Precios Reales vs. Predichos

In [ ]:
# Calcular los residuos
residuals = y_test - y_pred_rf

plt.figure(figsize=(10, 6))
sns.scatterplot(x=y_pred_rf, y=residuals, alpha=0.6)
plt.axhline(y=0, color='red', linestyle='--', linewidth=2, label='Error Cero')
plt.title('Gráfico de Residuos: Predicciones vs. Errores (Random Forest)')
plt.xlabel('Precio Predicho')
plt.ylabel('Residuos (Precio Real - Precio Predicho)')
plt.grid(True, linestyle=':', alpha=0.7)
plt.legend()
plt.show()

**De este gráfico podemos destacar lo siguiente:**

La mayoría de los puntos están concentrados alrededor de la línea roja central. Esto indica que, en promedio, el modelo no esta sesgado (no está prediciendo siempre muy alto o siempre muy bajo).

Se visualiza que en precios mas bajos el modelo es preciso, en precios del "medio" la nube de puntos se ensancha y en valores mas altos el modelo tiende a sobrestimar los precios.


In [ ]:
# Grafica de Precio Predicho vs. Precio Real

fig = px.scatter(x= y_pred_rf    , y=  y_test    ,
                 title='Precio predicho vs. precio real',
                 labels={'x':'Precio predicho','y':'Precio real'})
fig.show()

El siguiente grafico es complementario al anterior porque refleja que existe una correlación lineal positiva entre el precio predicho y el precio real.

Precios Bajos (< 200k): La nube es compacta y estrecha. Los puntos están pegados a la diagonal imaginaria. siendo el modelo es muy preciso aquí.

Precios Altos (> 300k): La nube se vuelve gorda y difusa. Para un valor real de 400k, el modelo a veces predice 300k y otras veces 480k.

Sabemos que el modelo esta topeado en USD 500k, lo cual es coherente con el tratamiento previo de datos,
El random forest predice promediando precios por lo cual nunca va a predecir valores que no haya visto (es decir por encima de USD 500k o por debajo de USD 40k)


# **6- Conclusiones Principales y Aplicación**

### **1. Conclusión del Modelo Predictivo**

El modelo de Random Forest Regressor, seleccionado tras el análisis comparativo de rendimiento, demuestra ser altamente efectivo para la predicción de precios de apartamentos en el mercado uruguayo, cumpliendo el objetivo central del proyecto.

**Rendimiento Clave:** Con un MAPE de 15.90% en la versión optimizada, el modelo ofrece un nivel de precisión significativamente útil para la tasación automática, manteniendo el error promedio de estimación en un nivel aceptable para la toma de decisiones en el sector inmobiliario.

**Principales Variables Explicativas:** El Feature Importance del modelo valida que los atributos que mayor peso tienen en la determinación del precio son consistentes con el conocimiento del mercado:

- COVERED_AREA: La variable más relevante, reflejando el valor intrínseco de la propiedad.

- BATHROOMS: Un indicador clave de la funcionalidad y el confort de la unidad.

- BEDROOMS: Un factor fundamental en la segmentación del mercado y el tamaño del hogar.

### **2. Próximos pasos para validación del modelo**

**Validación con datos recientes:**

Se sugiere realizar una prueba de validación sobre una muestra de casos más reciente en el tiempo. Con el objetivo de confirmar que el modelo mantiene su precisión, a pesar de las posibles fluctuaciones del mercado inmobiliario.


### **3. Aplicación del modelo**

**Uso Recomendado:**

El modelo podría ser implementado como un sistema de tasación automatizada en tiempo real. Esto permite a las inmobiliarias, agentes e inversores generar cotizaciones iniciales de forma instantánea y objetiva, mejorando la eficiencia operativa.

**Público Objetivo:**

Agencias Inmobiliarias: Para estandarizar y acelerar el proceso de captación de propiedades y fijación de precios iniciales.

Inversores Inmobiliarios: Como herramienta de filtrado rápido para identificar oportunidades de inversión y evaluar si una propiedad está subvalorada o sobrevalorada.

Valor Agregado: El uso del modelo reduce el sesgo humano en la tasación, proporciona una referencia objetiva y permite a la empresa escalar su capacidad de análisis de mercado, pasando de un enfoque manual a uno basado en Datos.